# **Scaling up RAG with Vector Databases**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Chunking Text for Retrieval-Augmented Generation Systems](#Chunking-Text-for-Retrieval-Augmented-Generation-Systems)
  - [Introduction](#Introduction)
  - [Understanding Document Chunking](#Understanding-Document-Chunking)
  - [Splitting Large Text into Chunks](#Splitting-Large-Text-into-Chunks)
  - [Inspecting the Input Data](#Inspecting-the-Input-Data)
  - [Adding Meaningful Metadata](#Adding-Meaningful-Metadata)
  - [Practical Usage Example](#Practical-Usage-Example)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
  - [Preserve Sentence Boundaries in Chunks](#Preserve-Sentence-Boundaries-in-Chunks)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chunking Text for Retrieval-Augmented Generation Systems

## Introduction

Welcome to the first lesson of our course on "**Scaling Up RAG with Vector Databases**"! This is the third course in our series, building on the foundations laid in the previous courses. Course 1 introduced a basic RAG pipeline, providing a foundational understanding of how retrieval and generation can be combined. Course 2 focused on text representation, with a particular emphasis on embeddings.

In this course, we'll focus on scaling your Retrieval-Augmented Generation (RAG) system by building and querying a vector database. You'll learn to preprocess documents, store chunk embeddings in ChromaDB, retrieve relevant chunks using advanced techniques like compound metadata filters and weighting, and construct prompts that can handle multiple context chunks. Additionally, we'll cover managing updates to your collection and large-scale ingestion using batch strategies.

Our journey begins with *document chunking*, a crucial preprocessing step that enhances the efficiency and effectiveness of vector databases in RAG systems. By the end of this lesson, you'll be able to break down a lengthy document into discrete segments, each tagged with essential metadata, paving the way for robust retrieval and storage in vector databases.

## Understanding Document Chunking

When dealing with Retrieval-Augmented Generation, you typically feed chunks of text — along with certain metadata — into downstream components, like embedding models or vector search engines. This **document chunking** process is crucial for several reasons:

1. **Manageability**: Smaller segments are easier to process. Many models have a maximum context length, meaning they can only handle a certain number of tokens at a time. If the input text exceeds this limit, the model may become inefficient or unable to process the request, leading to errors or truncated outputs. Additionally, even if a model can technically handle larger contexts, performance may degrade, resulting in slower processing times and reduced accuracy.

2. **Context Preservation**: A well-sized chunk still retains enough local context to be meaningful. Chunks should be neither too large (leading to potential memory issues) nor too small (risking the loss of context).

3. **Enhanced Retrieval**: When text is split rationally, you can retrieve only the relevant segments instead of searching through entire documents — this shortens query times and boosts accuracy.

Think of a real-world example: If you have a large reference manual, you wouldn't read the whole thing to answer a single question. Instead, you'd look for the exact section (or chunk) that pertains to your query. The same principle carries into text retrieval on a computer.

## Splitting Large Text into Chunks

Here's a sample function showcasing how to slice a long piece of text into parts of a specified word count:

In [1]:
def chunk_text(text, chunk_size=10):
    """
    Splits the given text into smaller chunks, each containing
    up to 'chunk_size' words. Returns a list of these chunk strings.
    """
    words = text.split()  # Tokenize by splitting on whitespace
    # Construct chunks by stepping through the words list in increments of chunk_size
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [2]:
sample_text = "This is a sample text that we will use to test our chunking function. It contains multiple sentences to make it more interesting."

chunks = chunk_text(sample_text)
for chunk in chunks:
    print(chunk)

This is a sample text that we will use to
test our chunking function. It contains multiple sentences to make
it more interesting.


After tokenizing the text, the function uses a list comprehension to iterate in steps of `chunk_size`. Each step produces a concise string of words that can be processed more easily by language models. Note that this method is a simplified approach and may not handle punctuation or sentence boundaries effectively, causing issues with *context preservation*; for more advanced chunking, consider using NLP libraries like NLTK or spaCy that can respect sentence boundaries.

## Inspecting the Input Data

Before moving on, let's briefly take a closer look at the dataset we'll be working with for chunking. Below is an example of two documents in JSON format:

```json
[
  {
    "id": 1,
    "content": "Hello world! This is a sample document used for testing chunk_text function."
  },
  {
    "id": 2,
    "content": "Another sample document. This is used for verifying the chunking of text in multiple documents. It includes additional sentences to provide a more comprehensive test case. By having a longer document, we can better assess how the chunking function performs when dealing with more extensive content."
  }
]
```

Our dataset consists of an array of items (or a list of dictionaries, in Python's terms), where each item has an `id` field that identifies the document and a `content` field containing the main text. After applying a chunking approach to each document, our data will be represented as smaller segments of text that are easier to process downstream, as we'll see later in the lesson.

## Adding Meaningful Metadata

So far, we have a method for chunking text, but in RAG systems, we often need to know *where* each piece came from and *what* it represents. That's where metadata comes in. Below is an example function that demonstrates how to loop through a structured dataset of documents, chunk each one, and store metadata for later use:

In [3]:
def load_and_chunk_dataset(data, chunk_size=10):
    """
    Iterates over a structured dataset of documents, splits each into chunks,
    and associates metadata (doc_id and chunk_id) with every piece.
    """
    all_chunks = []
    for doc in data:
        doc_id = doc["id"]
        doc_text = doc["content"]

        # Create smaller text segments from the original document
        doc_chunks = chunk_text(doc_text, chunk_size)

        # Label each chunk with its source identifier
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "text": chunk_str
                }
            )

    return all_chunks

Here's how it works step-by-step:

1. We iterate over `data`, which is a collection of different text entries in the dataset.
2. For each entry, the function extracts the `id` and `content` fields to process.
3. We apply the earlier `chunk_text` function to split the content into multiple pieces.
4. We then store relevant information — `doc_id` and `chunk_id` — along with the chunked text. This means every piece of text is *traceable* back to its origin, which can be crucial for retrieval and filtering in a vector database.

## Practical Usage Example

In this lesson, we'll use two sample documents to demonstrate the chunking process and to illustrate the concepts discussed. In the practice section, we'll be using a more realistic dataset for hands-on exercises; it can be found at `data/corpus.json`.

In [4]:
import json

# Open data/corpus.json file and load data
with open('data/corpus.json', 'r') as file:
    data = json.load(file)

chunked_docs = load_and_chunk_dataset(data, chunk_size=10)
print(f'Loaded and chunked {len(chunked_docs)} chunks from dataset.')
for doc in chunked_docs:
    print(doc)

Loaded and chunked 767 chunks from dataset.
{'doc_id': 1, 'chunk_id': 0, 'text': 'Artificial intelligence is transforming the way we approach complex problems'}
{'doc_id': 1, 'chunk_id': 1, 'text': 'in computing. Recent breakthroughs in machine learning have enabled faster'}
{'doc_id': 1, 'chunk_id': 2, 'text': 'data processing and smarter algorithms. The future of technology is'}
{'doc_id': 1, 'chunk_id': 3, 'text': 'expected to integrate AI into every facet of life.'}
{'doc_id': 2, 'chunk_id': 0, 'text': 'Robotics continues to advance with improved sensors and autonomous capabilities.'}
{'doc_id': 2, 'chunk_id': 1, 'text': 'Industrial applications are increasingly adopting these technologies for enhanced efficiency.'}
{'doc_id': 2, 'chunk_id': 2, 'text': 'Engineers remain optimistic about the seamless integration of robots in'}
{'doc_id': 2, 'chunk_id': 3, 'text': 'everyday manufacturing.'}
{'doc_id': 3, 'chunk_id': 0, 'text': 'Quantum computing has made significant strides in recent

This output clearly illustrates both the benefits and limitations of our simple chunking approach. While we've successfully broken down the documents into smaller pieces with proper metadata tracking, the quality of some chunks is problematic. For example, chunk 1 of document 1 contains only the fragment `'chunk_text function.'`, which lacks sufficient context to be meaningful on its own. Similarly, other chunks cut across sentence boundaries, creating segments that might be difficult for retrieval systems to properly interpret.

These issues highlight why more sophisticated chunking strategies that respect semantic boundaries (like sentences or paragraphs) are essential for **context preservation** in production RAG systems. Poor chunking can significantly reduce the effectiveness of retrieval, as chunks without adequate context may not match relevant queries or might provide incomplete information. In the practice section, we'll explore more advanced techniques that better preserve context and create more meaningful chunks.

## Conclusion and Next Steps

In this lesson, we focused on *why* and *how* to chunk larger text documents. We learned that chunking:

* Makes text blocks suitably sized for processing with language models.
* Preserves local context by grouping words carefully.
* Allows for the attachment of metadata that makes future retrieval more powerful.

Next, we'll embed these chunks and store them in a vector database so we can efficiently search within them. Feel free to **practice** splitting your own documents and confirm the results before moving on to the upcoming exercises. By mastering the chunking and metadata process, you're well on your way to creating scalable, retrieval-enhanced applications.

## Preserve Sentence Boundaries in Chunks

You've learned how to split text into chunks based on word count. Now, let's enhance your skills by refining the `chunk_text` function to better preserve sentence boundaries, which will help maintain the context and coherence of the text segments.

To achieve this, follow these steps:

* Modify the `chunk_text` function to split the text not only by word count but also by punctuation marks such as periods, exclamation points, and question marks. This ensures that sentences remain intact within chunks. *Hint*: you can use `re.split('[.!?]+', text)` to split `text` on multiple characters.

* After splitting by punctuation, group the resulting sentences into chunks. Add whole sentences to the current chunk while the total stays at or below `chunk_size` words; when the next sentence would push the count over `chunk_size`, close the current chunk and start a new one with that sentence. Never split a sentence across two chunks.

* If a single sentence has more than `chunk_size` words, keep it intact as its own chunk even though that chunk exceeds `chunk_size`: sentence boundaries take precedence over the word limit. With the default `chunk_size=30`, two documents in `corpus.json` start with a 31-word sentence, so a correct implementation prints exactly two chunks of 31 words and none longer.

Once you've implemented these changes, test your function with a sample text and observe how the chunks differ. Notice how preserving sentence boundaries can enhance the readability and context of each chunk. This exercise will deepen your understanding of text chunking and prepare you for more advanced techniques. Enjoy the challenge and happy coding!

For this and the next practices, we'll be using a dataset located at `data/corpus.json`. This dataset contains a collection of documents, each with fields such as `id`, `title`, `content`, `category`, `tags`, and `date`. Here's a sample entry from the file:

```json
{
  "id": 1,
  "title": "Revolutionizing Computing with AI",
  "content": "Artificial intelligence is transforming the way we approach complex problems in computing. Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms. The future of technology is expected to integrate AI into every facet of life.",
  "category": "Technology",
  "tags": ["AI", "machine learning", "computing", "innovation"],
  "date": "2025-02-01"
}
```

In [6]:
import os
import json
import re

def chunk_text(text, chunk_size=30):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of chunk strings.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    sentences = re.split('[.?!]+', text)

    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed
    chunks = []
    chunk_in_progress = False
    chunk_words = []
    chunk_word_count = 0
    for i, sentence in enumerate(sentences):
        if sentence:  # Only consider non-empty sentences in order to create only non-empty chunks
            words = sentence.split()
            # chunk_words += words
            # chunk_word_count += len(words)
            if chunk_word_count + len(words) >= chunk_size and not chunk_in_progress:  # keep sentence intact as its own chunk
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) < chunk_size and not chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) < chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) == chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) > chunk_size and chunk_in_progress:
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True

    if len(chunk_words):
        chunks.append(" ".join(chunk_words))

    return chunks


def load_and_chunk_dataset(file_path, chunk_size=30):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)
    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        doc_category = doc.get("category", "general")
        doc_chunks = chunk_text(doc_text, chunk_size)
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "category": doc_category,
                "text": chunk_str
            })
    return all_chunks

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")
chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=30)
print("Loaded and chunked", len(chunked_docs), "chunks from dataset.")
for c in chunked_docs:
    print(c)

Loaded and chunked 353 chunks from dataset.
{'doc_id': 0, 'chunk_id': 0, 'category': 'Technology', 'text': 'Artificial intelligence is transforming the way we approach complex problems in computing Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms'}
{'doc_id': 0, 'chunk_id': 1, 'category': 'Technology', 'text': 'The future of technology is expected to integrate AI into every facet of life'}
{'doc_id': 1, 'chunk_id': 0, 'category': 'Technology', 'text': 'Robotics continues to advance with improved sensors and autonomous capabilities Industrial applications are increasingly adopting these technologies for enhanced efficiency'}
{'doc_id': 1, 'chunk_id': 1, 'category': 'Technology', 'text': 'Engineers remain optimistic about the seamless integration of robots in everyday manufacturing'}
{'doc_id': 2, 'chunk_id': 0, 'category': 'Technology', 'text': 'Quantum computing has made significant strides in recent years, pushing the boundaries of wh

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[CodeSignal: Scaling up RAG with Vector Databases Course](https://codesignal.com/learn/courses/scaling-up-rag-with-vector-databases)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)